In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter, Language
from langchain_experimental.text_splitter import SemanticChunker
from dotenv import load_dotenv

In [1]:
%pip install chromadb 

^C


In [3]:
from langchain_core.documents import Document

# Create LangChain documents for IPL players

doc1 = Document(
        page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
        metadata={"team": "Royal Challengers Bangalore"}
    )
doc2 = Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
        metadata={"team": "Mumbai Indians"}
    )
doc3 = Document(
        page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
        metadata={"team": "Chennai Super Kings"}
    )
doc4 = Document(
        page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
        metadata={"team": "Mumbai Indians"}
    )
doc5 = Document(
        page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
        metadata={"team": "Chennai Super Kings"}
    )

In [4]:
docs = [doc1, doc2, doc3, doc4, doc5]   

In [5]:
embedder = GoogleGenerativeAIEmbeddings(model = 'gemini-embedding-2')

In [7]:
%pip install langchain-chroma chromadb


Note: you may need to restart the kernel to use updated packages.


In [10]:
from langchain_chroma import Chroma

vector_store = Chroma(
    embedding_function = embedder,
    persist_directory='my_chroma_db',
    collection_name='sample'    
)

In [11]:
vector_store.add_documents(documents=docs)

['d8f3c317-889f-4dad-bfe4-c51df360afda',
 '44224011-4507-4c5d-997e-1f34fc0d2402',
 '0f50ef1d-da07-4fb7-98c0-31a2d260c903',
 '25f6ac12-06bf-47e4-89cc-9852a656b3de',
 '76adfeb8-f110-4820-95f8-cfd95cd67263']

In [13]:
vector_store.get(include=['embeddings', 'documents', 'metadatas'])

{'ids': ['d5da57ae-47b9-42a6-a6c9-526b064e451f',
  '584811cf-78dd-4da3-84fb-864b4208e181',
  '2b20237a-8913-4ef7-9488-1f8c8eea0e21',
  '9d598765-1d5d-4341-a86d-12a72ec2820b',
  '930b9fbe-5433-4eed-9535-bc7301fc41d4',
  'd8f3c317-889f-4dad-bfe4-c51df360afda',
  '44224011-4507-4c5d-997e-1f34fc0d2402',
  '0f50ef1d-da07-4fb7-98c0-31a2d260c903',
  '25f6ac12-06bf-47e4-89cc-9852a656b3de',
  '76adfeb8-f110-4820-95f8-cfd95cd67263'],
 'embeddings': array([[ 0.0074764 ,  0.01034819,  0.01265225, ...,  0.00313006,
          0.01334844,  0.00201392],
        [ 0.00578349,  0.03275334, -0.00712996, ...,  0.00599   ,
          0.00842474,  0.01284763],
        [ 0.00278233,  0.02150997, -0.00733589, ..., -0.00771123,
          0.01547233,  0.01310577],
        ...,
        [ 0.00278233,  0.02150997, -0.00733589, ..., -0.00771123,
          0.01547233,  0.01310577],
        [ 0.03408059,  0.01119854, -0.01184934, ...,  0.00848975,
          0.02335239, -0.0077354 ],
        [-0.00296271,  0.01466555, 

In [14]:
# search documents
vector_store.similarity_search(
    query='Who among these are a bowler?',
    k=2
)

[Document(id='9d598765-1d5d-4341-a86d-12a72ec2820b', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
 Document(id='25f6ac12-06bf-47e4-89cc-9852a656b3de', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.')]

In [15]:
# search with similarity score
vector_store.similarity_search_with_score(
    query='Who among these are a bowler?',
    k=2
)

[(Document(id='9d598765-1d5d-4341-a86d-12a72ec2820b', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  0.6193535327911377),
 (Document(id='25f6ac12-06bf-47e4-89cc-9852a656b3de', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  0.6193535327911377)]

In [18]:
# meta-data filtering
vector_store.similarity_search_with_score(
    query="Batsman",
    filter={"team": "Chennai Super Kings"},
    k = 1
)

[(Document(id='76adfeb8-f110-4820-95f8-cfd95cd67263', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  0.7799780368804932)]

In [19]:
# update documents
updated_doc1 = Document(
    page_content="Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league. His ability to chase targets and anchor innings has made him one of the most dependable players in T20 cricket.",
    metadata={"team": "Royal Challengers Bangalore"}
)

vector_store.update_document(document_id='d5da57ae-47b9-42a6-a6c9-526b064e451f', document=updated_doc1)


In [20]:
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['d5da57ae-47b9-42a6-a6c9-526b064e451f',
  '584811cf-78dd-4da3-84fb-864b4208e181',
  '2b20237a-8913-4ef7-9488-1f8c8eea0e21',
  '9d598765-1d5d-4341-a86d-12a72ec2820b',
  '930b9fbe-5433-4eed-9535-bc7301fc41d4',
  'd8f3c317-889f-4dad-bfe4-c51df360afda',
  '44224011-4507-4c5d-997e-1f34fc0d2402',
  '0f50ef1d-da07-4fb7-98c0-31a2d260c903',
  '25f6ac12-06bf-47e4-89cc-9852a656b3de',
  '76adfeb8-f110-4820-95f8-cfd95cd67263'],
 'embeddings': array([[ 0.01820879,  0.0189696 ,  0.00767588, ..., -0.00233459,
          0.0060162 ,  0.01336217],
        [ 0.00578349,  0.03275334, -0.00712996, ...,  0.00599   ,
          0.00842474,  0.01284763],
        [ 0.00278233,  0.02150997, -0.00733589, ..., -0.00771123,
          0.01547233,  0.01310577],
        ...,
        [ 0.00278233,  0.02150997, -0.00733589, ..., -0.00771123,
          0.01547233,  0.01310577],
        [ 0.03408059,  0.01119854, -0.01184934, ...,  0.00848975,
          0.02335239, -0.0077354 ],
        [-0.00296271,  0.01466555, 

In [21]:
vector_store.delete(ids=['d5da57ae-47b9-42a6-a6c9-526b064e451f'])

In [22]:
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['584811cf-78dd-4da3-84fb-864b4208e181',
  '2b20237a-8913-4ef7-9488-1f8c8eea0e21',
  '9d598765-1d5d-4341-a86d-12a72ec2820b',
  '930b9fbe-5433-4eed-9535-bc7301fc41d4',
  'd8f3c317-889f-4dad-bfe4-c51df360afda',
  '44224011-4507-4c5d-997e-1f34fc0d2402',
  '0f50ef1d-da07-4fb7-98c0-31a2d260c903',
  '25f6ac12-06bf-47e4-89cc-9852a656b3de',
  '76adfeb8-f110-4820-95f8-cfd95cd67263'],
 'embeddings': array([[ 0.00578349,  0.03275334, -0.00712996, ...,  0.00599   ,
          0.00842474,  0.01284763],
        [ 0.00278233,  0.02150997, -0.00733589, ..., -0.00771123,
          0.01547233,  0.01310577],
        [ 0.03408059,  0.01119854, -0.01184934, ...,  0.00848975,
          0.02335239, -0.0077354 ],
        ...,
        [ 0.00278233,  0.02150997, -0.00733589, ..., -0.00771123,
          0.01547233,  0.01310577],
        [ 0.03408059,  0.01119854, -0.01184934, ...,  0.00848975,
          0.02335239, -0.0077354 ],
        [-0.00296271,  0.01466555, -0.01345706, ...,  0.00715038,
          0